In [1]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [2]:
# Import basic modules
import pandas as pd
from datum_api_client import DatumApi
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any


# Import warnings
import warnings
warnings.filterwarnings("ignore")
# pip install xlrd
# pip install openpyxl

In [3]:
from __future__ import annotations


def opendoor_stats_v1_exporter(
    input_path: str,
    *,
    output_onefile_jsonl: str = "OPENDOOR/onefile.jsonl",
    output_summary_csv: str = "OPENDOOR/summary.csv",
    output_best_params_jsonl: str = "OPENDOOR/best_params.jsonl",
    # snapshot times (hour, minute)
    entry_hm: tuple = (9, 20),
    entry_hm_earliest: tuple = (8, 50),   # вікно пошуку entry: остання точка в [earliest, entry_hm]
    exit_hm: dict = None,
    # bins Stack% і Bench% в entry
    stack_bin_min: float = -15.0,
    stack_bin_max: float = 15.0,
    stack_bin_step: float = 1.0,
    bench_bin_min: float = -15.0,
    bench_bin_max: float = 15.0,
    bench_bin_step: float = 1.0,
    # best params
    best_min_rate: float = 0.60,
    best_min_total: int = 10,
    # global filter
    min_events_per_ticker: int = 10,
    # column names
    BENCH_NUM_FIELD: str = "Bench%",
    STOCK_NUM_FIELD: str = "Stack%",
    # reading
    assume_sorted: bool = True,
    parquet_use_pyarrow: bool = True,
    csv_chunksize: int = 500_000,
    log_every_n_chunks: int = 5,
):
    """
    OpenDoor v1:
    - Snapshot Stack%/Bench% в entry_hm (default 9:20); якщо немає — бере останнє
      доступне значення у вікні [entry_hm_earliest, entry_hm] (default 08:50–09:20)
    - Snapshot Stack% в кожній exit точці: 5m(9:35), 10m(9:40), 20m(9:50), 30m(10:00)
    - move = Stack%_exit - Stack%_entry  →  long (>0) / short (<0)
    - Bins 1D: Stack%_entry, Bench%_entry  (окремо)
    - Bins 2D: Stack%_entry × Bench%_entry  (комбо)
    - best_params: rate >= best_min_rate і total >= best_min_total, stitch consecutive
    """
    import gc, json, time, math, gzip
    from collections import defaultdict, Counter
    from datetime import datetime
    import numpy as np
    import pandas as pd
    from pathlib import Path

    if exit_hm is None:
        exit_hm = {
            "5m":  (9, 35),
            "10m": (9, 40),
            "20m": (9, 50),
            "30m": (10, 0),
        }

    HORIZONS = list(exit_hm.keys())

    Path(output_onefile_jsonl).parent.mkdir(parents=True, exist_ok=True)
    Path(output_summary_csv).parent.mkdir(parents=True, exist_ok=True)
    Path(output_best_params_jsonl).parent.mkdir(parents=True, exist_ok=True)

    def _open_gz(path, mode="wt"):
        if str(path).lower().endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8", newline="\n", compresslevel=6)
        return open(path, mode.replace("t", ""), encoding="utf-8", newline="\n")

    summary_cols = (
        ["ticker", "bench", "events_total"] +
        [f"{h}_{d}" for h in HORIZONS for d in ("long_rate", "short_rate", "total")] +
        ["corr", "beta", "sigma"]
    )
    pd.DataFrame(columns=summary_cols).to_csv(output_summary_csv, index=False, mode="w")

    onefile_f     = _open_gz(output_onefile_jsonl, "wt")
    best_params_f = _open_gz(output_best_params_jsonl, "wt")
    best_params_f.write(json.dumps({
        "meta": {"version": "opendoor_v1", "generated_at": datetime.utcnow().isoformat() + "Z"}
    }) + "\n")

    # ── helpers ──────────────────────────────────────────────────────────────
    def _js(x):
        if x is None: return None
        if isinstance(x, (np.floating, float)):
            return None if (np.isnan(x) or np.isinf(x)) else float(x)
        if isinstance(x, (np.integer, int)): return int(x)
        if isinstance(x, (np.bool_, bool)): return bool(x)
        return x

    def _ok(x):
        try: return np.isfinite(float(x))
        except: return False

    def _clamp(v, lo, hi): return max(lo, min(hi, v))

    def _sbin(v, lo, hi, step):
        if not _ok(v): return None
        v = _clamp(float(v), lo, hi)
        return f"{round(math.floor(v / step) * step, 6):.1f}"

    def stack_bin(v): return _sbin(v, stack_bin_min, stack_bin_max, stack_bin_step)
    def bench_bin(v): return _sbin(v, bench_bin_min, bench_bin_max, bench_bin_step)
    def _score(rate, total): return float(rate) * math.log1p(int(total))

    # ── per-ticker state ──────────────────────────────────────────────────────
    cur_ticker = None
    cur_day    = None
    bench_seen = None
    static_set = False
    corr_s = beta_s = sigma_s = None

    day_entry = None   # (stack_pct, bench_pct) — остання валідна точка у вікні [earliest, entry_hm]
    day_exits = {}     # horizon -> stack_pct at exit time
    day_count = 0      # кількість днів з валідним entry snapshot

    counts        = {h: Counter() for h in HORIZONS}
    stack_bins_1d = {h: defaultdict(Counter) for h in HORIZONS}
    bench_bins_1d = {h: defaultdict(Counter) for h in HORIZONS}
    combo_bins_2d = {h: defaultdict(Counter) for h in HORIZONS}

    def _reset_ticker():
        nonlocal bench_seen, static_set, corr_s, beta_s, sigma_s
        nonlocal day_entry, day_exits, day_count
        bench_seen = None; static_set = False; corr_s = beta_s = sigma_s = None
        day_entry = None; day_exits = {}; day_count = 0
        for h in HORIZONS:
            counts[h].clear()
            stack_bins_1d[h].clear()
            bench_bins_1d[h].clear()
            combo_bins_2d[h].clear()

    def _reset_day():
        nonlocal day_entry, day_exits
        day_entry = None
        day_exits = {}

    def _finalize_day():
        nonlocal day_count
        if day_entry is None:
            return
        stack_920, bench_920 = day_entry
        sb = stack_bin(stack_920)
        bb = bench_bin(bench_920)
        day_count += 1

        for h in HORIZONS:
            exit_stack = day_exits.get(h)
            if exit_stack is None or not _ok(exit_stack):
                continue
            move = float(exit_stack) - float(stack_920)
            d = "long" if move > 0 else "short"

            counts[h]["total"] += 1
            counts[h][d] += 1

            if sb:
                stack_bins_1d[h][sb]["total"] += 1
                stack_bins_1d[h][sb][d] += 1

            if bb:
                bench_bins_1d[h][bb]["total"] += 1
                bench_bins_1d[h][bb][d] += 1

            if sb and bb:
                k = f"{sb}|{bb}"
                combo_bins_2d[h][k]["total"] += 1
                combo_bins_2d[h][k][d] += 1

    def _rates(c):
        tot = int(c.get("total", 0))
        lng = int(c.get("long", 0))
        sht = int(c.get("short", 0))
        return {
            "total": tot, "long": lng, "short": sht,
            "long_rate":  round(lng / tot, 4) if tot else None,
            "short_rate": round(sht / tot, 4) if tot else None,
        }

    def _best_1d(bins_d, direction, step):
        eligible = []
        for b_str, c in bins_d.items():
            tot = int(c.get("total", 0))
            if tot < best_min_total: continue
            cnt = int(c.get(direction, 0))
            rate = cnt / tot if tot else 0.0
            if rate >= best_min_rate:
                try: eligible.append((float(b_str), b_str, tot, cnt))
                except ValueError: pass
        eligible.sort()
        if not eligible: return []

        intervals = []
        lo_f, lo_s = eligible[0][0], eligible[0][1]
        hi_f, hi_s = eligible[0][0], eligible[0][1]
        agg = Counter({direction: eligible[0][3], "total": eligible[0][2]})

        for v, s, tot, cnt in eligible[1:]:
            if abs(v - (hi_f + step)) < 1e-9:
                hi_f, hi_s = v, s
                agg[direction] += cnt
                agg["total"] += tot
            else:
                intervals.append((lo_s, hi_s, dict(agg)))
                lo_f, lo_s, hi_f, hi_s = v, s, v, s
                agg = Counter({direction: cnt, "total": tot})
        intervals.append((lo_s, hi_s, dict(agg)))

        result = []
        for lo_s, hi_s, agg in intervals:
            tot = agg.get("total", 0)
            cnt = agg.get(direction, 0)
            rate = cnt / tot if tot else 0.0
            if tot >= best_min_total and rate >= best_min_rate:
                result.append({
                    "lo": lo_s, "hi": hi_s, "total": tot,
                    direction: cnt, "rate": round(rate, 4),
                    "score": round(_score(rate, tot), 4),
                })
        result.sort(key=lambda x: x["score"], reverse=True)
        return result

    def _best_2d(bins_d, direction, top_n=10):
        rows = []
        for key, c in bins_d.items():
            tot = int(c.get("total", 0))
            if tot < best_min_total: continue
            cnt = int(c.get(direction, 0))
            rate = cnt / tot if tot else 0.0
            if rate >= best_min_rate:
                parts = key.split("|")
                rows.append({
                    "stack_bin": parts[0] if len(parts) > 0 else None,
                    "bench_bin": parts[1] if len(parts) > 1 else None,
                    "total": tot, direction: cnt,
                    "rate": round(rate, 4),
                    "score": round(_score(rate, tot), 4),
                })
        rows.sort(key=lambda x: x["score"], reverse=True)
        return rows[:top_n]

    # ── flush ticker ──────────────────────────────────────────────────────────
    def _flush():
        if cur_ticker is None:
            return
        _finalize_day()

        events_total = max((int(counts[h].get("total", 0)) for h in HORIZONS), default=0)
        if events_total < min_events_per_ticker:
            _reset_ticker()
            return

        rates = {h: _rates(counts[h]) for h in HORIZONS}

        best = {}
        for h in HORIZONS:
            best[h] = {
                "stack_long":  _best_1d(stack_bins_1d[h], "long",  stack_bin_step),
                "stack_short": _best_1d(stack_bins_1d[h], "short", stack_bin_step),
                "bench_long":  _best_1d(bench_bins_1d[h], "long",  bench_bin_step),
                "bench_short": _best_1d(bench_bins_1d[h], "short", bench_bin_step),
                "combo_long":  _best_2d(combo_bins_2d[h], "long"),
                "combo_short": _best_2d(combo_bins_2d[h], "short"),
            }

        payload = {
            "ticker": cur_ticker,
            "bench": bench_seen,
            "static": {"corr": _js(corr_s), "beta": _js(beta_s), "sigma": _js(sigma_s)},
            "events_total": int(events_total),
            "days_with_entry": int(day_count),
            "params": {
                "entry_hm": list(entry_hm),
                "entry_hm_earliest": list(entry_hm_earliest),
                "exit_hm": {h: list(t) for h, t in exit_hm.items()},
                "stack_bins": {"min": stack_bin_min, "max": stack_bin_max, "step": stack_bin_step},
                "bench_bins": {"min": bench_bin_min, "max": bench_bin_max, "step": bench_bin_step},
                "best_min_rate": best_min_rate,
                "best_min_total": best_min_total,
            },
            "rates": {h: rates[h] for h in HORIZONS},
            "bins": {
                "stack_1d": {h: {b: dict(c) for b, c in stack_bins_1d[h].items()} for h in HORIZONS},
                "bench_1d": {h: {b: dict(c) for b, c in bench_bins_1d[h].items()} for h in HORIZONS},
                "combo_2d": {h: {k: dict(c) for k, c in combo_bins_2d[h].items()} for h in HORIZONS},
            },
            "best_params": best,
        }
        onefile_f.write(json.dumps(payload, ensure_ascii=False) + "\n")

        row = {"ticker": cur_ticker, "bench": bench_seen, "events_total": int(events_total)}
        for h in HORIZONS:
            r = rates[h]
            row[f"{h}_long_rate"]  = _js(r["long_rate"])
            row[f"{h}_short_rate"] = _js(r["short_rate"])
            row[f"{h}_total"]      = int(r["total"])
        row.update({"corr": _js(corr_s), "beta": _js(beta_s), "sigma": _js(sigma_s)})
        pd.DataFrame([row], columns=summary_cols).to_csv(output_summary_csv, mode="a", header=False, index=False)

        best_params_f.write(json.dumps(
            {"ticker": cur_ticker, "bench": bench_seen, "best": best}, ensure_ascii=False
        ) + "\n")

        _reset_ticker()

    # ── chunk processor ───────────────────────────────────────────────────────
    def _process_chunk(chunk, ci):
        nonlocal cur_ticker, cur_day, bench_seen, static_set, corr_s, beta_s, sigma_s, day_entry

        req = {"ticker", "date", "dt"}
        if not req.issubset(chunk.columns):
            raise KeyError(f"Missing columns: {sorted(req - set(chunk.columns))}")

        if not assume_sorted:
            chunk["dt"] = pd.to_datetime(chunk["dt"], errors="coerce", utc=True)
            chunk.sort_values(["ticker", "date", "dt"], inplace=True)

        def _col(name):
            return chunk[name] if name in chunk.columns else pd.Series(np.nan, index=chunk.index)

        s_dt = pd.to_datetime(_col("dt"), errors="coerce", utc=True)
        ok   = s_dt.notna().to_numpy(copy=False)
        if not ok.any(): return

        s_dt2  = s_dt[ok]
        h_arr  = s_dt2.dt.hour.to_numpy(dtype="int16", copy=False)
        m_arr  = s_dt2.dt.minute.to_numpy(dtype="int16", copy=False)
        tk_arr = _col("ticker")[ok].to_numpy(copy=False)
        ds_arr = _col("date")[ok].to_numpy(copy=False)

        stock_arr = pd.to_numeric(_col(STOCK_NUM_FIELD)[ok], errors="coerce").to_numpy(dtype="float64", copy=False)
        bench_arr = pd.to_numeric(_col(BENCH_NUM_FIELD)[ok], errors="coerce").to_numpy(dtype="float64", copy=False)

        bn_arr    = _col("bench")[ok].to_numpy(copy=False)  if "bench" in chunk.columns else None
        corr_arr  = _col("corr")[ok].to_numpy(copy=False)   if "corr"  in chunk.columns else None
        beta_arr  = _col("beta")[ok].to_numpy(copy=False)   if "beta"  in chunk.columns else None
        sigma_arr = _col("sigma")[ok].to_numpy(copy=False)  if "sigma" in chunk.columns else None

        for i in range(len(tk_arr)):
            tk   = tk_arr[i]
            ds   = ds_arr[i]
            t    = (int(h_arr[i]), int(m_arr[i]))
            spct = float(stock_arr[i])
            bpct = float(bench_arr[i])

            # ticker boundary
            if cur_ticker is not None and tk != cur_ticker:
                _flush()
                cur_ticker = tk; cur_day = ds
                _reset_day()

            if cur_ticker is None:
                cur_ticker = tk; cur_day = ds

            # static fields
            if bn_arr is not None and bench_seen is None:
                v = bn_arr[i]
                if pd.notna(v) and str(v).strip():
                    bench_seen = str(v)

            if not static_set and corr_arr is not None and beta_arr is not None and sigma_arr is not None:
                c, b, s = corr_arr[i], beta_arr[i], sigma_arr[i]
                if pd.notna(c) and pd.notna(b) and pd.notna(s):
                    corr_s, beta_s, sigma_s = float(c), float(b), float(s)
                    static_set = True

            # day boundary
            if ds != cur_day:
                _finalize_day()
                cur_day = ds
                _reset_day()

            # entry window: постійно оновлюємо до останньої валідної точки у [earliest, entry_hm]
            if entry_hm_earliest <= t <= entry_hm and _ok(spct):
                day_entry = (spct, bpct if _ok(bpct) else float("nan"))

            # exit snapshots
            for h, xt in exit_hm.items():
                if t == xt and h not in day_exits and _ok(spct):
                    day_exits[h] = spct

    # ── main read loop ────────────────────────────────────────────────────────
    t0 = time.time()
    total_rows = 0
    last_rows  = 0
    last_ts    = t0
    is_parquet = str(input_path).lower().endswith((".parquet", ".pq", ".parq"))

    print(f"START OpenDoor v1  file={input_path}  parquet={is_parquet}")
    print(f"  entry window={entry_hm_earliest}..{entry_hm}  exits={exit_hm}  min_events={min_events_per_ticker}")

    try:
        if is_parquet and parquet_use_pyarrow:
            import pyarrow.parquet as pq
            pf = pq.ParquetFile(input_path)
            wanted = ["ticker", "date", "dt", "bench", "corr", "beta", "sigma",
                      STOCK_NUM_FIELD, BENCH_NUM_FIELD]
            cols = [c for c in wanted if c in pf.schema.names]

            for ci in range(pf.num_row_groups):
                chunk = pf.read_row_group(ci, columns=cols).to_pandas()
                _process_chunk(chunk, ci + 1)
                total_rows += len(chunk)
                if (ci + 1) % log_every_n_chunks == 0:
                    now = time.time()
                    rps = (total_rows - last_rows) / max(now - last_ts, 1e-6)
                    print(f"[rg {ci+1:>4}/{pf.num_row_groups}] rows={total_rows:,} speed={rps:,.0f}/s elapsed={now-t0:.1f}s")
                    last_rows, last_ts = total_rows, now
                del chunk
                if (ci + 1) % 20 == 0: gc.collect()

        elif not is_parquet:
            for ci, chunk in enumerate(
                pd.read_csv(input_path, compression="infer", low_memory=False, chunksize=csv_chunksize), 1
            ):
                _process_chunk(chunk, ci)
                total_rows += len(chunk)
                if ci % log_every_n_chunks == 0:
                    now = time.time()
                    rps = (total_rows - last_rows) / max(now - last_ts, 1e-6)
                    print(f"[chunk {ci}] rows={total_rows:,} speed={rps:,.0f}/s elapsed={now-t0:.1f}s")
                    last_rows, last_ts = total_rows, now
                del chunk
                if ci % 20 == 0: gc.collect()

        else:
            df = pd.read_parquet(input_path)
            step = 1_000_000
            for ci, start in enumerate(range(0, len(df), step), 1):
                _process_chunk(df.iloc[start:start + step], ci)
                total_rows += len(df.iloc[start:start + step])

        _flush()
        elapsed = time.time() - t0
        print(f"DONE rows={total_rows:,} elapsed={elapsed:.1f}s")
        print(f"  onefile     = {output_onefile_jsonl}")
        print(f"  summary     = {output_summary_csv}")
        print(f"  best_params = {output_best_params_jsonl}")

    finally:
        onefile_f.close()
        best_params_f.close()


In [4]:
from pathlib import Path
import os


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path   = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve()   if sig_env   else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break

        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")

        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")

    return final_path, out_dir


# ── runner ────────────────────────────────────────────────────────────────────

FINAL_PATH, OUT_DIR = _resolve_orion_paths("opendoor")

opendoor_stats_v1_exporter(
    input_path=str(FINAL_PATH),
    output_onefile_jsonl=str(OUT_DIR / "onefile.jsonl.gz"),
    output_best_params_jsonl=str(OUT_DIR / "best_params.jsonl.gz"),
    output_summary_csv=str(OUT_DIR / "summary.csv"),
    entry_hm=(9, 20),
    exit_hm={"5m": (9, 35), "10m": (9, 40), "20m": (9, 50), "30m": (10, 0)},
    stack_bin_min=-15.0, stack_bin_max=15.0, stack_bin_step=1.0,
    bench_bin_min=-15.0, bench_bin_max=15.0, bench_bin_step=1.0,
    best_min_rate=0.60, best_min_total=10, min_events_per_ticker=10,
    assume_sorted=True,
)


START OpenDoor v1  file=C:\datum-api-examples-main\OriON\CRACEN\final.parquet  parquet=True
  entry window=(8, 50)..(9, 20)  exits={'5m': (9, 35), '10m': (9, 40), '20m': (9, 50), '30m': (10, 0)}  min_events=10
[rg    5/7527] rows=53,286 speed=281,342/s elapsed=0.2s


[rg   10/7527] rows=99,590 speed=816,405/s elapsed=0.2s
[rg   15/7527] rows=222,843 speed=1,074,758/s elapsed=0.4s
[rg   20/7527] rows=310,176 speed=910,022/s elapsed=0.5s


[rg   25/7527] rows=329,585 speed=453,416/s elapsed=0.5s
[rg   30/7527] rows=397,093 speed=314,108/s elapsed=0.7s


[rg   35/7527] rows=471,042 speed=758,582/s elapsed=0.8s
[rg   40/7527] rows=501,304 speed=794,626/s elapsed=0.9s
[rg   45/7527] rows=591,651 speed=834,268/s elapsed=1.0s
[rg   50/7527] rows=640,659 speed=799,223/s elapsed=1.0s


[rg   55/7527] rows=693,309 speed=622,295/s elapsed=1.1s
[rg   60/7527] rows=742,460 speed=747,179/s elapsed=1.2s
[rg   65/7527] rows=766,172 speed=465,264/s elapsed=1.2s
[rg   70/7527] rows=857,147 speed=1,002,661/s elapsed=1.3s


...

[rg 7470/7527] rows=77,127,626 speed=605,588/s elapsed=305.3s
[rg 7475/7527] rows=77,168,899 speed=572,464/s elapsed=305.4s


[rg 7480/7527] rows=77,224,725 speed=210,195/s elapsed=305.7s


[rg 7485/7527] rows=77,281,940 speed=192,056/s elapsed=306.0s
[rg 7490/7527] rows=77,323,495 speed=594,646/s elapsed=306.0s
[rg 7495/7527] rows=77,390,709 speed=771,582/s elapsed=306.1s


[rg 7500/7527] rows=77,441,449 speed=916,375/s elapsed=306.2s
[rg 7505/7527] rows=77,487,095 speed=693,573/s elapsed=306.2s
[rg 7510/7527] rows=77,534,827 speed=772,589/s elapsed=306.3s


[rg 7515/7527] rows=77,605,295 speed=623,959/s elapsed=306.4s
[rg 7520/7527] rows=77,650,579 speed=891,190/s elapsed=306.5s
[rg 7525/7527] rows=77,705,095 speed=751,814/s elapsed=306.5s
DONE rows=77,717,692 elapsed=306.6s
  onefile     = C:\datum-api-examples-main\OriON\signals\opendoor\onefile.jsonl.gz
  summary     = C:\datum-api-examples-main\OriON\signals\opendoor\summary.csv
  best_params = C:\datum-api-examples-main\OriON\signals\opendoor\best_params.jsonl.gz
